# Modelagem e Otimização de Modelos de Classificação

## Objetivo
Este notebook tem como objetivo aplicar diferentes algoritmos de classificação para o problema de predição de inadimplência, utilizando técnicas de otimização de hiperparâmetros com validação cruzada.

---

## Ferramentas e Bibliotecas

- **pandas** → utilizado para manipulação e análise dos dados  
- **train_test_split** → responsável por dividir os dados em conjuntos de treino e teste  
- **GridSearchCV** → utilizado para encontrar a melhor combinação de hiperparâmetros para cada modelo, utilizando validação cruzada  

---

## Modelos Utilizados

Serão avaliados os seguintes algoritmos de classificação:

- Regressão Logística (*Logistic Regression*)  
- Árvore de Decisão (*Decision Tree*)  
- Random Forest
- XGBoost
- SVM
- KNN 

---

## Estratégia

Para cada modelo:
- Será definido um conjunto reduzido de hiperparâmetros  
- Será aplicado o **GridSearchCV** com validação cruzada  
- O desempenho será avaliado utilizando métrica ROC-AUC

---

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

2. Leitura e preparação dos dados
- Selecionamos apenas as colunas relevantes para o problema de crédito.
- Mantemos apenas Pessoa Física (PF)
- Converte a coluna ¨carteira_vencida string para número
- Transforma ¨carteira_vencida" em problema binário:

In [9]:
data = pd.read_csv("scrdata_202505.csv", sep=";")

data = data[[
    "uf", "cliente", "cnae_ocupacao",
    "porte", "modalidade", "submodalidade",
    "carteira_vencida"
]]

data = data[data["cliente"] == "PF"]

data["carteira_vencida"] = data["carteira_vencida"].str.replace(",", ".").astype(float)

data["carteira_vencida"] = (data["carteira_vencida"] > 0).astype(int)

Separação entre X e y e  Tratamento de variáveis categóricas
X → variáveis explicativas
y → variável alvo

- Removemos cliente por ser apenas identificador.
- Identificamos colunas de texto
- Aplicamos [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) (necessário porque modelos do scikit-learn só aceitam números)
- drop_first=True evita multicolinearidade


In [10]:
X = data.drop(["carteira_vencida", "cliente"], axis=1)
y = data["carteira_vencida"]
# separar X e y
# dados de entrada (features)
# y = variável alvo (target)
X = data.drop(["carteira_vencida", "cliente"], axis=1)
categorical_cols = X.select_dtypes(include=["object", "string"]).columns

# Essa função vem do pandas e serve para transformar variáveis categóricas (texto) em variáveis numéricas (0/1).
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
y = data["carteira_vencida"]

Divisão treino e teste
- 70% treino / 30% teste
- random_state=42 garante reprodutibilidade

! O teste só é usado no final (avaliação real).

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## Modelos utilizados
``` python
models = {
    "logistic_regression": LogisticRegression(),
    "decision_tree": DecisionTreeClassifier(),
    "random_forest": RandomForestClassifier(),
    "xgboost": XGBClassifier(),
    "svm": SVC(),
    "knn": KNeighborsClassifier()
}


## Regressão Logística

A [Regressão Logística](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) é um modelo de classificação supervisionada amplamente utilizado em problemas binários.

O modelo funciona a partir de uma combinação linear das variáveis de entrada, produzindo um valor que é transformado em probabilidade por meio da função logística (sigmoid), resultando em valores entre 0 e 1.

### Hiperparâmetros utilizados

- **C**: controla a regularização do modelo.
  - Valores menores (ex: 0.01) → modelo mais simples (mais regularização)
  - Valores maiores (ex: 10) → modelo mais complexo (menos regularização)
  - Os valores foram escolhidos em escala logarítmica para explorar diferentes ordens de magnitude.

- **solver**: algoritmo de otimização utilizado para encontrar os coeficientes do modelo.
  - `lbfgs`: bom desempenho geral, especialmente para datasets maiores (default)
  - `liblinear`: eficiente para problemas binários e datasets menores

### Estratégia

Foi utilizado o GridSearchCV com validação cruzada (5 folds) para identificar a melhor combinação de hiperparâmetros, utilizando a métrica ROC AUC, adequada para problemas com classes desbalanceadas.

In [12]:
param_grid_lr = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs", "liblinear"]
}

In [ ]:
grid_lr = GridSearchCV(
    estimator=LogisticRegression(),
    param_grid=param_grid_lr,
    cv=5,
    scoring="roc_auc", # ideal para dados desbalanceados
)

In [14]:
grid_lr.fit(X_train, y_train)

print(grid_lr.best_params_)
print(grid_lr.best_score_)

best_lr = grid_lr.best_estimator_

{'C': 10, 'solver': 'liblinear'}
0.7508917204073774


## Árvore de Decisão

A [Árvore de Decisão](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) é um modelo baseado em regras, que realiza divisões sucessivas nos dados com o objetivo de separar as classes.

Cada divisão é feita com base em uma variável e um ponto de corte, formando uma estrutura semelhante a uma árvore.

Esse modelo é interpretável e capaz de capturar relações não lineares entre as variáveis.

### Hiperparâmetros utilizados

- **max_depth**: define a profundidade máxima da árvore.
  - Valores menores → modelo mais simples (menos overfitting)
  - Valor `None` → árvore cresce livremente

- **min_samples_split**: número mínimo de amostras necessárias para dividir um nó.
  - Valores maiores tornam o modelo mais conservador

- **min_samples_leaf**: número mínimo de amostras em uma folha.
  - Evita que a árvore crie divisões muito específicas

- **criterion**: métrica utilizada para avaliar a qualidade da divisão.
  - `gini`: índice de Gini (default)
  - `entropy`: ganho de informação

### Estratégia

Foi utilizado o GridSearchCV com validação cruzada (5 folds) para ajustar os hiperparâmetros e reduzir o risco de overfitting.

In [15]:
param_grid_dt = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"]
}

In [ ]:
grid_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_dt,
    cv=5,
    scoring="roc_auc",
)

In [17]:
grid_dt.fit(X_train, y_train)

print(grid_dt.best_params_)
print(grid_dt.best_score_)

best_dt = grid_dt.best_estimator_

Fitting 5 folds for each of 72 candidates, totalling 360 fits
{'criterion': 'entropy', 'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 10}
0.7308856847552466


## Random Forest

O [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) é um modelo de ensemble que combina múltiplas árvores de decisão para melhorar a performance e reduzir o overfitting.

Cada árvore é treinada em uma amostra diferente dos dados, e a decisão final é feita por votação.

Esse modelo tende a ser mais robusto e apresentar melhor generalização em comparação com uma única árvore.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores na floresta.
  - Mais árvores → melhor desempenho (até certo ponto)

- **max_depth**: profundidade máxima das árvores.
  - Controla o overfitting

- **min_samples_split**: mínimo de amostras para dividir um nó

- **min_samples_leaf**: O número mínimo de amostras necessário para que um nó seja uma folha

### Estratégia

Foi utilizado GridSearchCV para encontrar a melhor combinação de hiperparâmetros, utilizando validação cruzada e a métrica ROC AUC.

In [18]:
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [ ]:
grid_rf = GridSearchCV(
    estimator= RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring="roc_auc",
)

In [20]:
grid_rf.fit(X_train, y_train)
print("Melhores parâmetros:", grid_rf.best_params_)
print("Melhor score:", grid_rf.best_score_)
best_rf = grid_rf.best_estimator_


Fitting 5 folds for each of 24 candidates, totalling 120 fits
Melhores parâmetros: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Melhor score: 0.7558092707530493


## XGBoost

O [XGBoost](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html) (Extreme Gradient Boosting) é um modelo baseado em árvores que utiliza a técnica de boosting, onde múltiplos modelos fracos são combinados sequencialmente para formar um modelo forte.

A cada nova árvore, o modelo tenta corrigir os erros das árvores anteriores, melhorando progressivamente o desempenho.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores no modelo
  - Mais árvores → melhor aprendizado (até certo limite)

- **learning_rate**: taxa de aprendizado
  - Valores menores tornam o aprendizado mais lento, porém mais robusto

- **max_depth**: profundidade das árvores
  - Controla a complexidade do modelo

### Estratégia

Foi utilizado GridSearchCV com validação cruzada (5 folds) e métrica ROC AUC para encontrar a melhor combinação de hiperparâmetros.

In [ ]:
param_grid_xgb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1],
    "max_depth": [3, 6, 10]
}

grid_xgb = GridSearchCV(
    estimator=XGBClassifier(eval_metric="logloss"),
    param_grid=param_grid_xgb,
    cv=5,
    scoring="roc_auc",
)

grid_xgb.fit(X_train, y_train)

print("Melhores parâmetros:", grid_xgb.best_params_)
print("Melhor score:", grid_xgb.best_score_)

best_xgb = grid_xgb.best_estimator_

Fitting 5 folds for each of 12 candidates, totalling 60 fits


/home/milena-hamerski/Documents/UTFPR/TCC/analise_ds/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [22:56:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/milena-hamerski/Documents/UTFPR/TCC/analise_ds/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [22:56:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/milena-hamerski/Documents/UTFPR/TCC/analise_ds/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [22:56:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/milena-hamerski/Documents/UTFPR/TCC/analise_ds/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [22:56:32] WARNING: /__w/

Melhores parâmetros: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
Melhor score: 0.7631912660250609


## Support Vector Machine (SVM)

O [SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) é um modelo que busca encontrar o melhor hiperplano que separa as classes no espaço de características.

Ele pode utilizar diferentes funções de kernel para lidar com relações lineares e não lineares.

### Hiperparâmetros utilizados

- **C**: controla a margem de separação
  - Valores altos → menor margem (mais complexo)
  - Valores baixos → maior margem (mais simples)

- **kernel**: define o tipo de separação
  - `linear`: separação linear
  - `rbf`: separação não linear (default)

### Estratégia

Foi utilizado GridSearchCV com validação cruzada para identificar a melhor combinação de parâmetros, utilizando a métrica ROC AUC.

In [ ]:
param_grid_svm = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"]
}

grid_svm = GridSearchCV(
    estimator=SVC(probability=True),
    param_grid=param_grid_svm,
    cv=5,
    scoring="roc_auc",
)

grid_svm.fit(X_train, y_train)

print("Melhores parâmetros:", grid_svm.best_params_)
print("Melhor score:", grid_svm.best_score_)

best_svm = grid_svm.best_estimator_

Fitting 5 folds for each of 6 candidates, totalling 30 fits


## K-Nearest Neighbors (KNN)

O [KNN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) é um algoritmo baseado em instâncias, que classifica uma nova observação com base nos seus vizinhos mais próximos.

A classe é definida pela maioria dos vizinhos.

### Hiperparâmetros utilizados

- **n_neighbors**: número de vizinhos considerados (default = 5)
  - Valores menores → modelo mais sensível 
  - Valores maiores → modelo mais estável

- **weights**: forma de ponderação
  - `uniform`: todos os vizinhos têm o mesmo peso (uniform)
  - `distance`: vizinhos mais próximos têm maior peso

### Estratégia

Foi utilizado GridSearchCV com validação cruzada para encontrar os melhores parâmetros, utilizando ROC AUC como métrica.

In [ ]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"]
}

grid_knn = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid_knn,
    cv=5,
    scoring="roc_auc",
)

grid_knn.fit(X_train, y_train)

print("Melhores parâmetros:", grid_knn.best_params_)
print("Melhor score:", grid_knn.best_score_)

best_knn = grid_knn.best_estimator_